# L48 · 🎨 时频图解 — STFT 与 Mel 谱可视化

**目标**：用图建立时频分析的完整视觉记忆——分帧、STFT热力图、Mel滤波器组、log-mel。

🔗 Aurora连接：`aurora.audio.stft`（`stft`, `magnitude_spectrogram`） · `aurora.audio.mel`（`mel_filterbank`, `mel_spectrogram`）

STFT 的本质是：把一段连续信号切成短帧，对每帧做 DFT，把所有帧的频谱横向拼成一张热力图。Mel 滤波器组再在频率轴上做一次三角形加权平均，把人耳感知不到的高频细节压缩掉，让同样维度装进更多低频信息。这两步串联就是音频 AI 最常见的前端特征管道。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import chirp
from aurora.audio.stft import stft, magnitude_spectrogram
from aurora.audio.mel import mel_spectrogram, mel_filterbank

## 1. 分帧：滑动窗口在信号上移动

对采样率 `sr` 的信号，帧长 `n_fft` 个采样点，帧移 `hop_length` 个采样点。第 `t` 帧的起始样本索引为 `t * hop_length`，该帧覆盖区间 `[t*hop_length, t*hop_length + n_fft)`。

帧之间的重叠率 = `1 - hop_length / n_fft`。典型设置：`n_fft=1024, hop_length=256`，重叠 75%。

窗函数（Hann）`w[n] = 0.5 * (1 - cos(2π n / (N-1)))` 让帧边缘平滑降到 0，抑制频谱泄漏。

In [ ]:
# 演示分帧（静态多帧重叠示意）
sr = 8000
duration = 0.05  # 50ms，便于可视化
t = np.arange(int(sr * duration)) / sr
signal = np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 1200 * t)

n_fft = 256
hop_length = 64
n_frames_show = 4

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(t, signal, color='0.6', lw=1, label='原始信号')

colors = plt.cm.tab10(np.linspace(0, 0.4, n_frames_show))
for i in range(n_frames_show):
    start = i * hop_length
    end = start + n_fft
    if end > len(signal):
        break
    hann = 0.5 * (1 - np.cos(2 * np.pi * np.arange(n_fft) / (n_fft - 1)))
    frame_t = t[start:end]
    ax.fill_between(frame_t, signal[start:end] * hann,
                    alpha=0.35, color=colors[i], label=f'帧 {i}')
    ax.axvline(frame_t[0], color=colors[i], lw=0.8, ls='--')

ax.set_xlabel('时间 (s)')
ax.set_title(f'分帧示意  n_fft={n_fft}  hop_length={hop_length}  重叠={1-hop_length/n_fft:.0%}')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 2. Mel 滤波器组：线性频率轴 vs Mel 轴

Mel 刻度的转换公式：`mel = 2595 * log10(1 + f / 700)`。在线性频率轴上，等间距的 Mel 滤波器中心频率**低频密、高频疏**；换到 Mel 轴上看才是均匀分布。

每个三角形滤波器 `H_m(k)` 覆盖相邻两个 Mel 中心之间的频率区间，面积归一化使各滤波器能量可比。通常取 `n_mels=80` 个滤波器，覆盖 `[fmin, fmax]`，比如 `fmin=0, fmax=sr/2`。

In [ ]:
# 对比：线性频率轴 vs Mel轴下的滤波器分布
sr = 22050
n_fft = 1024
n_mels = 40
fmax = sr // 2

fb = mel_filterbank(n_mels=n_mels, n_fft=n_fft, sample_rate=sr, fmin=0.0, fmax=fmax)
# fb shape: (n_mels, n_fft//2 + 1)
freqs = np.linspace(0, fmax, n_fft // 2 + 1)

def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：线性频率轴
for m in range(n_mels):
    axes[0].plot(freqs, fb[m], lw=0.9)
axes[0].set_xlabel('频率 (Hz)')
axes[0].set_ylabel('权重')
axes[0].set_title(f'线性频率轴  n_mels={n_mels}\n（低频密、高频疏）')

# 右图：Mel频率轴
mel_freqs = hz_to_mel(freqs)
for m in range(n_mels):
    axes[1].plot(mel_freqs, fb[m], lw=0.9)
axes[1].set_xlabel('频率 (Mel)')
axes[1].set_ylabel('权重')
axes[1].set_title(f'Mel 频率轴  n_mels={n_mels}\n（均匀分布）')

plt.tight_layout()
plt.show()

## 3. 参数实验：chirp 信号的线性频谱 vs Mel 频谱

**信号**：`chirp(sr=22050, duration=2.0, f0=200, f1=8000)` — 线性升频扫描。

**关键参数**：
- `n_fft=1024`，`hop_length=256`：控制时频分辨率权衡
- `n_mels=80`：Mel 通道数，越多高频细节越多
- `fmax=8000`：截断上限，超过此频率的能量被丢弃

**预期现象**：
- 线性频谱图：升频轨迹在高频区**占据更多行**，低频细节被压缩
- Mel 频谱图：升频轨迹从下到上**速度均匀**，低频信息更丰富
- log-mel 动态范围约 40-60 dB，噪底更清晰

In [ ]:
sr = 22050
n_fft = 1024
hop_length = 256
n_mels = 80
fmax = 8000

sig = chirp(sample_rate=sr, duration=2.0, f0=200, f1=fmax)

# 线性频谱图
S = magnitude_spectrogram(sig, n_fft=n_fft, hop_length=hop_length)
# aurora stft 输出 (n_frames, n_freqs)，转置为 (n_freqs, n_frames) 以便绘图
S_db = (20 * np.log10(S + 1e-8)).T

# Mel 频谱图（log scale）
M = mel_spectrogram(sig, sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
                    n_mels=n_mels, fmin=0.0, fmax=fmax)
M_db = (10 * np.log10(M + 1e-8)).T  # (n_frames, n_mels) → (n_mels, n_frames)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

times = np.arange(S.shape[0]) * hop_length / sr  # S.shape[0]=n_frames
freqs_lin = np.linspace(0, sr / 2, S.shape[1])       # S.shape[1]=n_freqs

im0 = axes[0].pcolormesh(times, freqs_lin / 1000, S_db,
                         shading='auto', cmap='magma',
                         vmin=S_db.max() - 60)
axes[0].set_xlabel('时间 (s)')
axes[0].set_ylabel('频率 (kHz)')
axes[0].set_title('线性幅度频谱图 (dB)\n高频占大量行数')
plt.colorbar(im0, ax=axes[0], label='dB')

times_mel = np.arange(M.shape[0]) * hop_length / sr  # M.shape[0]=n_frames
im1 = axes[1].pcolormesh(times_mel, np.arange(n_mels), M_db,
                         shading='auto', cmap='magma',
                         vmin=M_db.max() - 60)
axes[1].set_xlabel('时间 (s)')
axes[1].set_ylabel('Mel 通道索引')
axes[1].set_title(f'Log-Mel 频谱图  n_mels={n_mels}\n升频轨迹均匀上升')
plt.colorbar(im1, ax=axes[1], label='dB')

plt.tight_layout()
plt.show()

print(f'aurora 线性频谱 shape: {S.shape}  (n_frames, n_fft//2+1)')
print(f'aurora Mel 频谱  shape: {M.shape}  (n_frames, n_mels)')

## 本课小结

`stft` + `magnitude_spectrogram` 输出形状 `(n_fft//2+1, n_frames)` 的线性幅度频谱；`mel_filterbank` 将其压缩为 `(n_mels, n_frames)` 的 Mel 频谱，两者均来自 `aurora.audio.stft` 和 `aurora.audio.mel` 模块。Mel 轴在感知上均匀，使低频细节在有限维度下得到更好保留。下一节（`x3_mfcc.ipynb`）将在 log-Mel 基础上做 DCT，得到 MFCC 倒谱系数。